**TÍTULO**

F3N6 — Ondas de Kelvin oceânicas: extração, rastreamento e velocidade de propagação

**CONTEXTO**

Ondas de Kelvin equatoriais descendentes transportam o sinal de recarga do Pacífico oeste para leste, precedendo o aquecimento de superfície em Niño 3.4 (Cui et al., 2025). Sua assinatura aparece como anomalias positivas de altura da superfície do mar (SSHA) e aprofundamento da termoclina propagando para leste a ~2–3 m/s.

**DESAFIO**

Hipótese: pulsos de SSHA filtrada propagam-se para leste com velocidade compatível com o primeiro modo baroclínico (~2,7 m/s) e antecedem os eventos do catálogo F3N2. Objetivos: (i) extrair o sinal de Kelvin da SSHA equatorial UFS+GLORYS (120°E–280°E); (ii) rastrear pulsos ao longo da bacia; (iii) estimar a velocidade de fase por correlação defasada entre longitudes.

**METODOLOGIA**

SSH semanal médio 2°S–2°N por longitude; anomalia por climatologia semanal por longitude; filtro espacial: remoção da média zonal instantânea e suavização em longitude (~10°) para reter escalas de onda longas. Pulsos: excedências de +1,5σ na longitude de referência 160°E. Velocidade: lag de correlação máxima entre a referência e cada longitude a leste; regressão distância × lag fornece a velocidade média (m/s), com a série d20 do master como verificação da resposta da termoclina.

**RESULTADOS ESPERADOS**

TabF3N6_ssha_filtrada.csv (matriz lon × tempo), TabF3N6_pulsos_kelvin.csv, TabF3N6_velocidade_fase.csv; FigF3N6_hovmoller_ssha_filtrada (janela recente), FigF3N6_lag_distancia (ajuste da velocidade) e FigF3N6_pulsos_vs_d20.

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. CUI, Y. et al. Oceanic Kelvin waves and their role in ENSO evolution. Journal of Geophysical Research: Oceans, 2025. DOI: https://doi.org/10.1029/2025JC023275
2. MEINEN, C. S.; McPHADEN, M. J. Observations of Warm Water Volume Changes in the Equatorial Pacific and Their Relationship to El Niño and La Niña. Journal of Climate, v. 13, p. 3551-3559, 2000. DOI: https://doi.org/10.1175/1520-0442(2000)013<3551:OOWWVC>2.0.CO;2

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
from nino_brasil import fase3_nino as f3

f3.ensure_dirs()
master = f3.load_master_weekly()
fisicas = [c for c in master.columns if c != 'ocean_source_code']
print(f'Master semanal F2: {master.shape[0]} semanas x {len(fisicas)} variaveis fisicas'
      f" | simulado={master.attrs.get('dado_simulado', False)}")
print(f'Periodo: {master.index.min().date()} a {master.index.max().date()}')

Master semanal F2: 2376 semanas x 44 variaveis fisicas | simulado=False
Periodo: 1981-01-04 a 2026-07-12


In [2]:
ssh = f3.load_ssh_equatorial_weekly()
ssha = f3.lon_anomaly_matrix(ssh)
kelvin = f3.kelvin_bandpass(ssha, smooth_deg=10.0)
f3.save_table(kelvin, 'TabF3N6_ssha_filtrada')
print(f'SSHA filtrada: {kelvin.shape}; simulado={ssh.attrs.get("dado_simulado", False)}')

[tabela] TabF3N6_ssha_filtrada.csv (2377x641) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
SSHA filtrada: (2377, 641); simulado=False


In [3]:
referencia_lon = 160.0
lons = np.array([float(c) for c in kelvin.columns])
col_ref = kelvin.columns[int(np.argmin(np.abs(lons - referencia_lon)))]
serie_ref = kelvin[col_ref]
sigma = serie_ref.std(ddof=1)
pulsos = serie_ref[serie_ref > 1.5 * sigma]
grupos = (pulsos.index.to_series().diff() > pd.Timedelta(weeks=4)).cumsum()
catalogo_pulsos = pulsos.groupby(grupos).agg(['idxmax', 'max', 'count'])
catalogo_pulsos.columns = ['semana_pico_pulso', 'ssha_pico_m', 'duracao_semanas']
f3.save_table(catalogo_pulsos.reset_index(drop=True), 'TabF3N6_pulsos_kelvin', index=False)
print(f'{len(catalogo_pulsos)} pulsos de Kelvin > 1,5 sigma em {referencia_lon}E')

[tabela] TabF3N6_pulsos_kelvin.csv (20x3) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
20 pulsos de Kelvin > 1,5 sigma em 160.0E


In [4]:
lags = f3.phase_speed_from_lags(kelvin, reference_lon=referencia_lon, max_lag_weeks=10)
ajuste = f3.fit_phase_speed(lags)
f3.save_table(lags.assign(**ajuste), 'TabF3N6_velocidade_fase', index=False)
print(f"velocidade de fase media: {ajuste['velocidade_m_s']:.2f} m/s (r2={ajuste['r2']:.2f});"
      ' referencia teorica c1 ~ 2,7 m/s')

fig, ax = plt.subplots(figsize=(8, 5))
validos = lags[lags['r_max'] >= 0.3]
ax.scatter(validos['lag_otimo_semanas'], validos['distancia_km'], s=14)
if np.isfinite(ajuste['velocidade_m_s']):
    xx = np.linspace(0, validos['lag_otimo_semanas'].max() + 1, 20)
    ax.plot(xx, ajuste['velocidade_m_s'] * xx * 7 * 86400 / 1000, color='crimson',
            label=f"{ajuste['velocidade_m_s']:.2f} m/s")
ax.set_xlabel('Lag otimo (semanas) vs 160E')
ax.set_ylabel('Distancia para leste (km)')
ax.set_title('Propagacao para leste do sinal de Kelvin (SSHA filtrada)')
ax.legend()
fig.tight_layout()
f3.save_figure(fig, 'FigF3N6_lag_distancia')
plt.close(fig)

[tabela] TabF3N6_velocidade_fase.csv (641x7) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
velocidade de fase media: 1.45 m/s (r2=0.30); referencia teorica c1 ~ 2,7 m/s


[figura] FigF3N6_lag_distancia.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3


In [5]:
janela = kelvin.loc[kelvin.index.max() - pd.Timedelta(weeks=200):]
fig, ax = plt.subplots(figsize=(9, 10))
malha = ax.pcolormesh(np.array([float(c) for c in janela.columns]), janela.index,
                      janela.to_numpy(), cmap='RdBu_r',
                      vmin=-np.nanmax(np.abs(janela.to_numpy())),
                      vmax=np.nanmax(np.abs(janela.to_numpy())))
ax.set_xlabel('Longitude (E)')
ax.set_title('Hovmoller da SSHA filtrada (assinatura de Kelvin) — ultimas ~200 semanas')
fig.colorbar(malha, ax=ax, label='SSHA filtrada (m)')
fig.tight_layout()
f3.save_figure(fig, 'FigF3N6_hovmoller_ssha_filtrada')
plt.close(fig)

if 'd20_m' in master:
    d20 = f3.zscore(f3.weekly_anomaly(pd.to_numeric(master['d20_m'], errors='coerce')))
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(serie_ref.index, serie_ref / sigma, lw=0.8, label=f'SSHA filtrada {referencia_lon}E (sigma)')
    ax.plot(d20.index, d20, lw=0.8, label='anomalia D20 Nino 3.4 (z)')
    for semana in catalogo_pulsos['semana_pico_pulso']:
        ax.axvline(pd.Timestamp(semana), color='orange', lw=0.5, alpha=0.6)
    ax.legend(); ax.set_title('Pulsos de Kelvin e resposta da termoclina (D20)')
    fig.tight_layout()
    f3.save_table(pd.DataFrame({'ssha_ref_sigma': serie_ref / sigma, 'd20_z': d20}),
                  'FigF3N6_pulsos_vs_d20_dados')
    f3.save_figure(fig, 'FigF3N6_pulsos_vs_d20')
    plt.close(fig)

[figura] FigF3N6_hovmoller_ssha_filtrada.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] FigF3N6_pulsos_vs_d20_dados.csv (2377x2) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N6_pulsos_vs_d20.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
